In [171]:
from __future__ import print_function
import os
import glob
import json
import argparse
import time
import codecs
from collections import defaultdict
from sklearn.svm import SVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer
from sklearn import preprocessing
from sklearn.calibration import CalibratedClassifierCV
from nltk.stem import PorterStemmer
import string
import numpy as np

import nltk


from textblob import TextBlob


In [203]:
# def represent_text(text,n):
#     # Extracts all character 'n'-grams from  a 'text'
#     if n>0:
#         tokens = [text[i:i+n] for i in range(len(text)-n+1)]
#     frequency = defaultdict(int)
#     for token in tokens:
#         frequency[token] += 1
#     return frequency

def read_files(path,label):
    # Reads all text files located in the 'path' and assigns them to 'label' class
    files = glob.glob(path+os.sep+label+os.sep+'*.txt')
    texts=[]
    for i,v in enumerate(files):
        f=codecs.open(v,'r',encoding='utf-8')
        texts.append((f.read(),label))
        f.close()
    return texts

def extract_vocabulary(texts,n,ft):
    # Extracts all characer 'n'-grams occurring at least 'ft' times in a set of 'texts'
    occurrences=defaultdict(int)
    for (text,label) in texts:
        text_occurrences=represent_text(text,n)
        for ngram in text_occurrences:
            if ngram in occurrences:
                occurrences[ngram]+=text_occurrences[ngram]
            else:
                occurrences[ngram]=text_occurrences[ngram]
    vocabulary=[]
    for i in occurrences.keys():
        if occurrences[i]>=ft:
            vocabulary.append(i)
    return vocabulary

def represent_text(text, n):
    # Extracts all character 'n'-grams and POS 'n'-grams from a 'text'
    tokens = []
    if n > 0:
        tokens.extend([text[i:i+n] for i in range(len(text)-n+1)])

    # Extract syntactic features
    syntax = defaultdict(int)
    for sent in nltk.sent_tokenize(text):
        for chunk in nltk.ne_chunk(nltk.pos_tag(nltk.word_tokenize(sent))):
            if hasattr(chunk, 'label'):
                syntax[chunk.label()] += 1

    # Add POS n-grams to tokens
    pos_tags = nltk.pos_tag(nltk.word_tokenize(text))
    pos_tags = [pos for _, pos in pos_tags]
    if n > 0:
        tokens.extend(["_".join(pos_tags[i:i+n]) for i in range(len(pos_tags)-n+1)])

    # Add sentiment feature
    blob = TextBlob(text)
    sentiment = blob.sentiment.polarity
    tokens.append(f'sentiment_{sentiment:.2f}')

    # Combine all features
    features = defaultdict(int)
    features.update(syntax)
    for token in tokens:
        features[token] += 1

    return features


In [204]:
def get_pos_tags(text):
    tokens = nltk.word_tokenize(text)
    pos_tags = nltk.pos_tag(tokens)
    pos_tagged_text = ' '.join([tag for word, tag in pos_tags])
    return pos_tagged_text


def get_sentiment_score(text):
    blob = TextBlob(text)
    sentiment_score = blob.sentiment.polarity
    return f"sentiment_{sentiment_score:.2f}"

In [205]:
def read_files(path, label):
    filelist = glob.glob(path + os.sep + label + os.sep + '*.txt')
    documents = []
    for filename in filelist:
        with open(filename, 'r', encoding='utf-8') as f:
            text = f.read()
            pos_tags = get_pos_tags(text)
            sentiment_score = get_sentiment_score(text)
            text_with_features = f"{text} {pos_tags} {sentiment_score}"
            documents.append((text_with_features, label))
    return documents

In [179]:
def remove_punctuations_from_docs(train_docs):
    """
    Remove punctuations from documents in the train_docs list.
    Args:
        train_docs (list): List of tuples where each tuple contains a document and its author.
    Returns:
        list: List of tuples where each tuple contains the document with punctuations removed and its author.
    """
    # Get punctuation list
    punctuations = set(string.punctuation+'–')
    
    # List to store documents with punctuations removed
    docs_without_punctuations = []
    
    for doc, author in train_docs:
        # Remove punctuations from document
        doc_without_punctuations = ''.join([char for char in doc if char not in punctuations])
        
        # Append tuple of document with punctuations removed and its author to docs_without_punctuations list
        docs_without_punctuations.append((doc_without_punctuations, author))
    
    return docs_without_punctuations

def case_folding(train_docs):

    
    # List to store documents with punctuations removed
    docs_lowercase = []
    
    for doc, author in train_docs:
        # Remove punctuations from document
        doc_lower = doc.lower()
        
        # Append tuple of document with punctuations removed and its author to docs_without_punctuations list
        docs_lowercase.append((doc_lower, author))
    
    return docs_lowercase

def stem_documents(train_docs):
    """
    Apply PorterStemmer to each document in train_docs list.
    Args:
        train_docs (list): List of tuples where each tuple contains a document and its author.
    Returns:
        list: List of tuples where each tuple contains a stemmed document and its author.
    """
    stemmed_docs = []
    porter = PorterStemmer()

    for doc, author in train_docs:
        words = doc.split()  # Split document into words
        stemmed_words = [porter.stem(word) for word in words]  # Apply PorterStemmer to each word
        stemmed_doc = ' '.join(stemmed_words)  # Merge stemmed words by putting spaces between them
        stemmed_docs.append((stemmed_doc, author))  # Append stemmed document and author to stemmed_docs list

    return stemmed_docs

def remove_stopwords_from_docs(train_docs,stop_words):
    """
    Remove stopwords from documents in the train_docs list.
    Args:
        train_docs (list): List of tuples where each tuple contains a document and its author.
    Returns:
        list: List of tuples where each tuple contains the document with stopwords removed and its author.
    """
    
    # List to store documents with stopwords removed
    docs_without_stopwords = []
    
    for doc, author in train_docs:
        # Tokenize document into words
        words = doc.split()
        
        # Remove stopwords
        words_filtered = [word for word in words if word.lower() not in stop_words]
        
        # Join filtered words back into a single document string
        doc_without_stopwords = ' '.join(words_filtered)
        
        # Append tuple of document with stopwords removed and its author to docs_without_stopwords list
        docs_without_stopwords.append((doc_without_stopwords, author))
    
    return docs_without_stopwords

In [180]:
path = 'C:\\Users\\90538\\Downloads\\pan19-cross-domain-authorship-attribution-training-dataset-2019-01-23\\pan19-cross-domain-authorship-attribution-training-dataset-2019-01-23'
outpath = "C:\\Users\\90538\\Downloads\\outputs"
stop_words = ['a', 'all', 'an', 'and', 'any', 'are', 'as', 'be', 'been', 'but', 'by', 'few', 'for', 'have', 'he', 'her', 'here', 'him', 'his', 'how', 'i', 'in', 'is', 'it', 'its', 'many', 'me', 'my', 'none', 'of', 'on', 'or', 'our', 'she', 'some', 'the', 'their', 'them', 'there', 'they', 'that', 'this', 'us', 'was', 'what', 'when', 'where', 'which', 'who', 'why', 'will', 'with', 'you', 'your']

In [209]:
import os
import glob
import json
from sklearn.neighbors import KNeighborsClassifier
from sklearn import preprocessing
from sklearn.calibration import CalibratedClassifierCV
## KNN
def KNN_algo(path,outpath,n=4,ft=5,pt=0.01):
    start_time = time.time()
    # Reading information about the collection
    infocollection = path+os.sep+'collection-info.json'
    problems = []
    language = []
    with open(infocollection, 'r') as f:
        for attrib in json.load(f):
            problems.append(attrib['problem-name'])
            language.append(attrib['language'])
    for index,problem in enumerate(problems[0:5]):
        print(problem)
        # Reading information about the problem
        infoproblem = path+os.sep+problem+os.sep+'problem-info.json'
        candidates = []
        with open(infoproblem, 'r') as f:
            fj = json.load(f)
            unk_folder = fj['unknown-folder']
            for attrib in fj['candidate-authors']:
                candidates.append(attrib['author-name'])
        # Building training set
        train_docs=[]
        for candidate in candidates:
            train_docs.extend(read_files(path+os.sep+problem,candidate))
        train_docs = remove_punctuations_from_docs(train_docs)         
        train_docs = stem_documents(train_docs)
        train_docs = remove_stopwords_from_docs(train_docs, stop_words)
        train_docs = case_folding(train_docs)
        train_texts = [text for i,(text,label) in enumerate(train_docs)]
        train_labels = [label for i,(text,label) in enumerate(train_docs)]
        vocabulary = extract_vocabulary(train_docs,n,ft)
        vectorizer = CountVectorizer(analyzer='char',ngram_range=(n,n),lowercase=False,vocabulary=vocabulary)
        train_data = vectorizer.fit_transform(train_texts)
        train_data = train_data.astype(float)
        for i,v in enumerate(train_texts):
            train_data[i]=train_data[i]/len(train_texts[i])
        print('\t', 'language: ', language[index])
        print('\t', len(candidates), 'candidate authors')
        print('\t', len(train_texts), 'known texts')
        print('\t', 'vocabulary size:', len(vocabulary))
        # Building test set
        test_docs=read_files(path+os.sep+problem,unk_folder)
        test_docs = remove_punctuations_from_docs(test_docs)
        test_docs = stem_documents(test_docs)
        test_docs = remove_stopwords_from_docs(test_docs, stop_words)
        test_docs = case_folding(test_docs)
        test_texts = [text for i,(text,label) in enumerate(test_docs)]
        test_data = vectorizer.transform(test_texts)
        test_data = test_data.astype(float)
        for i,v in enumerate(test_texts):
            test_data[i]=test_data[i]/len(test_texts[i])
        print('\t', len(test_texts), 'unknown texts')
        



        # Parameters
        k = 6 # Number of nearest neighbors
        pt = 0.05  # Probability threshold for rejection

        # Scaling data
        max_abs_scaler = preprocessing.MaxAbsScaler()
        scaled_train_data = max_abs_scaler.fit_transform(train_data)
        scaled_test_data = max_abs_scaler.transform(test_data)

        # Training classifier
        clf = CalibratedClassifierCV(KNeighborsClassifier(n_neighbors=k))
        clf.fit(scaled_train_data, train_labels)

        # Making predictions
        predictions = clf.predict(scaled_test_data)
        proba = clf.predict_proba(scaled_test_data)

        # Reject option (used in open-set cases)
        count = 0
        for i, p in enumerate(predictions):
            sproba = sorted(proba[i], reverse=True)
            if sproba[0] - sproba[1] < pt:
                predictions[i] = u'<UNK>'
                count += 1
        print('\t', count, 'texts left unattributed')

        # Saving output data
        out_data = []
        unk_filelist = glob.glob(path + os.sep + problem + os.sep + unk_folder + os.sep + '*.txt')
        pathlen = len(path + os.sep + problem + os.sep + unk_folder + os.sep)
        for i, v in enumerate(predictions):
            out_data.append({'unknown-text': unk_filelist[i][pathlen:], 'predicted-author': v})
        with open(outpath + os.sep + 'answers-' + problem + '.json', 'w') as f:
            json.dump(out_data, f, indent=4)

            
        print('\t', 'answers saved to file','answers-'+problem+'.json')
    print('elapsed time:', time.time() - start_time)

In [181]:
## Single SVM
def baseline(path,outpath,n=3,ft=5,pt=0.1):
    start_time = time.time()
    # Reading information about the collection
    infocollection = path+os.sep+'collection-info.json'
    problems = []
    language = []
    with open(infocollection, 'r') as f:
        for attrib in json.load(f):
            problems.append(attrib['problem-name'])
            language.append(attrib['language'])
    for index,problem in enumerate(problems[0:5]):
        print(problem)
        # Reading information about the problem
        infoproblem = path+os.sep+problem+os.sep+'problem-info.json'
        candidates = []
        with open(infoproblem, 'r') as f:
            fj = json.load(f)
            unk_folder = fj['unknown-folder']
            for attrib in fj['candidate-authors']:
                candidates.append(attrib['author-name'])
        # Building training set
        train_docs=[]
        for candidate in candidates:
            train_docs.extend(read_files(path+os.sep+problem,candidate))
        train_docs = remove_punctuations_from_docs(train_docs)
        train_docs = stem_documents(train_docs)
        train_docs = remove_stopwords_from_docs(train_docs, stop_words)
        train_texts = [text for i,(text,label) in enumerate(train_docs)]
        train_labels = [label for i,(text,label) in enumerate(train_docs)]
        vocabulary = extract_vocabulary(train_docs,n,ft)
        vectorizer = CountVectorizer(analyzer='char',ngram_range=(n,n),lowercase=False,vocabulary=vocabulary)
        train_data = vectorizer.fit_transform(train_texts)
        train_data = train_data.astype(float)
        for i,v in enumerate(train_texts):
            train_data[i]=train_data[i]/len(train_texts[i])
        print('\t', 'language: ', language[index])
        print('\t', len(candidates), 'candidate authors')
        print('\t', len(train_texts), 'known texts')
        print('\t', 'vocabulary size:', len(vocabulary))
        # Building test set
        test_docs=read_files(path+os.sep+problem,unk_folder)
        test_docs = remove_punctuations_from_docs(test_docs)
        test_docs = stem_documents(test_docs)
        test_docs = remove_stopwords_from_docs(test_docs, stop_words)
        test_texts = [text for i,(text,label) in enumerate(test_docs)]
        test_data = vectorizer.transform(test_texts)
        test_data = test_data.astype(float)
        for i,v in enumerate(test_texts):
            test_data[i]=test_data[i]/len(test_texts[i])
        print('\t', len(test_texts), 'unknown texts')
        
        # Applying SVM
        max_abs_scaler = preprocessing.MaxAbsScaler()
        scaled_train_data = max_abs_scaler.fit_transform(train_data)
        scaled_test_data = max_abs_scaler.transform(test_data)
        
        clf=CalibratedClassifierCV(OneVsRestClassifier(SVC(C=1)))
        clf.fit(scaled_train_data, train_labels)
        predictions=clf.predict(scaled_test_data)
        proba=clf.predict_proba(scaled_test_data)
        # Reject option (used in open-set cases)
        count=0
        for i,p in enumerate(predictions):
            sproba=sorted(proba[i],reverse=True)
            if sproba[0]-sproba[1]<pt:
                predictions[i]=u'<UNK>'
                count=count+1
        print('\t',count,'texts left unattributed')
        # Saving output data
        out_data=[]
        unk_filelist = glob.glob(path+os.sep+problem+os.sep+unk_folder+os.sep+'*.txt')
        pathlen=len(path+os.sep+problem+os.sep+unk_folder+os.sep)
        for i,v in enumerate(predictions):
            out_data.append({'unknown-text': unk_filelist[i][pathlen:], 'predicted-author': v})
        with open(outpath+os.sep+'answers-'+problem+'.json', 'w') as f:
            json.dump(out_data, f, indent=4)
            
        print('\t', 'answers saved to file','answers-'+problem+'.json')
    print('elapsed time:', time.time() - start_time)

In [200]:
from sklearn.svm import SVC
from sklearn.preprocessing import MaxAbsScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.ensemble import BaggingClassifier
import glob
import os
import json
def baseline_bagging(path,outpath,n=4,ft=5,pt=0.01):
    start_time = time.time()
    # Reading information about the collection
    infocollection = path+os.sep+'collection-info.json'
    problems = []
    language = []
    with open(infocollection, 'r') as f:
        for attrib in json.load(f):
            problems.append(attrib['problem-name'])
            language.append(attrib['language'])
    for index,problem in enumerate(problems[0:5]):
        print(problem)
        # Reading information about the problem
        infoproblem = path+os.sep+problem+os.sep+'problem-info.json'
        candidates = []
        with open(infoproblem, 'r') as f:
            fj = json.load(f)
            unk_folder = fj['unknown-folder']
            for attrib in fj['candidate-authors']:
                candidates.append(attrib['author-name'])
        # Building training set
        train_docs=[]
        for candidate in candidates:
            train_docs.extend(read_files(path+os.sep+problem,candidate))
        train_docs = remove_punctuations_from_docs(train_docs)
        train_docs = stem_documents(train_docs)
        train_docs = remove_stopwords_from_docs(train_docs, stop_words)
        train_texts = [text for i,(text,label) in enumerate(train_docs)]
        train_labels = [label for i,(text,label) in enumerate(train_docs)]
        vocabulary = extract_vocabulary(train_docs,n,ft)
        vectorizer = CountVectorizer(analyzer='char',ngram_range=(n,n),lowercase=False,vocabulary=vocabulary)
        train_data = vectorizer.fit_transform(train_texts)
        train_data = train_data.astype(float)
        for i,v in enumerate(train_texts):
            train_data[i]=train_data[i]/len(train_texts[i])
        print('\t', 'language: ', language[index])
        print('\t', len(candidates), 'candidate authors')
        print('\t', len(train_texts), 'known texts')
        print('\t', 'vocabulary size:', len(vocabulary))
        # Building test set
        test_docs=read_files(path+os.sep+problem,unk_folder)
        test_docs = remove_punctuations_from_docs(test_docs)
        test_docs = stem_documents(test_docs)
        test_docs = remove_stopwords_from_docs(test_docs, stop_words)
        test_texts = [text for i,(text,label) in enumerate(test_docs)]
        test_data = vectorizer.transform(test_texts)
        test_data = test_data.astype(float)
        for i,v in enumerate(test_texts):
            test_data[i]=test_data[i]/len(test_texts[i])
        print('\t', len(test_texts), 'unknown texts')
        
        # Applying SVM with bagging
        # Scale the data
        max_abs_scaler = MaxAbsScaler()
        scaled_train_data = max_abs_scaler.fit_transform(train_data)
        scaled_test_data = max_abs_scaler.transform(test_data)

        # Create a bagging classifier with SVM base estimator
        clf = CalibratedClassifierCV(BaggingClassifier(SVC(C=1), n_estimators=10, random_state=42))
        clf.fit(scaled_train_data, train_labels)

        # Make predictions and reject option
        predictions = clf.predict(scaled_test_data)
        proba = clf.predict_proba(scaled_test_data)
        count = 0
        for i, p in enumerate(predictions):
            sproba = sorted(proba[i], reverse=True)
            if sproba[0] - sproba[1] < pt:
                predictions[i] = u'<UNK>'
                count += 1
        print('\t', count, 'texts left unattributed')

        # Save output data
        out_data = []
        unk_filelist = glob.glob(path + os.sep + problem + os.sep + unk_folder + os.sep + '*.txt')
        pathlen = len(path + os.sep + problem + os.sep + unk_folder + os.sep)
        for i, v in enumerate(predictions):
            out_data.append({'unknown-text': unk_filelist[i][pathlen:], 'predicted-author': v})
        with open(outpath + os.sep + 'answers-' + problem + '.json', 'w') as f:
            json.dump(out_data, f, indent=4)




        print('\t', 'answers saved to file','answers-'+problem+'.json')
    print('elapsed time:', time.time() - start_time)

In [220]:
from sklearn.ensemble import RandomForestClassifier
from sklearn import preprocessing
from sklearn.calibration import CalibratedClassifierCV
import glob
import os
import json

def random_forest(path,outpath,n=4,ft=5,pt=0.01):
    start_time = time.time()
    # Reading information about the collection
    infocollection = path+os.sep+'collection-info.json'
    problems = []
    language = []
    with open(infocollection, 'r') as f:
        for attrib in json.load(f):
            problems.append(attrib['problem-name'])
            language.append(attrib['language'])
    for index,problem in enumerate(problems[0:5]):
        print(problem)
        # Reading information about the problem
        infoproblem = path+os.sep+problem+os.sep+'problem-info.json'
        candidates = []
        with open(infoproblem, 'r') as f:
            fj = json.load(f)
            unk_folder = fj['unknown-folder']
            for attrib in fj['candidate-authors']:
                candidates.append(attrib['author-name'])
        # Building training set
        train_docs=[]
        for candidate in candidates:
            train_docs.extend(read_files(path+os.sep+problem,candidate))
        #train_docs = remove_punctuations_from_docs(train_docs)
        #train_docs = stem_documents(train_docs)
        train_docs = remove_stopwords_from_docs(train_docs, stop_words)
        #train_docs = case_folding(train_docs)
        train_texts = [text for i,(text,label) in enumerate(train_docs)]
        train_labels = [label for i,(text,label) in enumerate(train_docs)]
        vocabulary = extract_vocabulary(train_docs,n,ft)
        vectorizer = CountVectorizer(analyzer='char',ngram_range=(n,n),lowercase=False,vocabulary=vocabulary)
        train_data = vectorizer.fit_transform(train_texts)
        train_data = train_data.astype(float)
        for i,v in enumerate(train_texts):
            train_data[i]=train_data[i]/len(train_texts[i])
        print('\t', 'language: ', language[index])
        print('\t', len(candidates), 'candidate authors')
        print('\t', len(train_texts), 'known texts')
        print('\t', 'vocabulary size:', len(vocabulary))
        # Building test set
        test_docs=read_files(path+os.sep+problem,unk_folder)
        #test_docs = remove_punctuations_from_docs(test_docs)
        #test_docs = stem_documents(test_docs)
        test_docs = remove_stopwords_from_docs(test_docs, stop_words)
        #test_docs = case_folding(test_docs)
        test_texts = [text for i,(text,label) in enumerate(test_docs)]
        test_data = vectorizer.transform(test_texts)
        test_data = test_data.astype(float)
        for i,v in enumerate(test_texts):
            test_data[i]=test_data[i]/len(test_texts[i])
        print('\t', len(test_texts), 'unknown texts')
        

        # Random Forest
        # Scale the data
        max_abs_scaler = MaxAbsScaler()
        scaled_train_data = max_abs_scaler.fit_transform(train_data)
        scaled_test_data = max_abs_scaler.transform(test_data)
        
        n_est = 100
        max_depth = 20
        # Create a Random Forest classifier
        rf = RandomForestClassifier(n_estimators=n_est, random_state=42)
        clf = CalibratedClassifierCV(rf)

        # Train the classifier
        clf.fit(scaled_train_data, train_labels)

        # Make predictions and reject option
        predictions = clf.predict(scaled_test_data)
        proba = clf.predict_proba(scaled_test_data)
        count = 0
        for i, p in enumerate(predictions):
            sproba = sorted(proba[i], reverse=True)
            if sproba[0] - sproba[1] < pt:
                predictions[i] = u'<UNK>'
                count += 1
        print('\t', count, 'texts left unattributed')

        # Save output data
        out_data = []
        unk_filelist = glob.glob(path + os.sep + problem + os.sep + unk_folder + os.sep + '*.txt')
        pathlen = len(path + os.sep + problem + os.sep + unk_folder + os.sep)
        for i, v in enumerate(predictions):
            out_data.append({'unknown-text': unk_filelist[i][pathlen:], 'predicted-author': v})
        with open(outpath + os.sep + 'answers-' + problem + '.json', 'w') as f:
            json.dump(out_data, f, indent=4)


        print('\t', 'answers saved to file','answers-'+problem+'.json')
    print('elapsed time:', time.time() - start_time)

In [221]:
baseline_boosting_kernel(path,outpath)

problem00001
	 language:  en
	 9 candidate authors
	 63 known texts
	 vocabulary size: 12739
	 561 unknown texts
	 17 texts left unattributed
	 answers saved to file answers-problem00001.json
problem00002
	 language:  en
	 9 candidate authors
	 63 known texts
	 vocabulary size: 12969
	 137 unknown texts
	 12 texts left unattributed
	 answers saved to file answers-problem00002.json
problem00003
	 language:  en
	 9 candidate authors
	 63 known texts
	 vocabulary size: 12610
	 211 unknown texts
	 17 texts left unattributed
	 answers saved to file answers-problem00003.json
problem00004
	 language:  en
	 9 candidate authors
	 63 known texts
	 vocabulary size: 13017
	 273 unknown texts
	 33 texts left unattributed
	 answers saved to file answers-problem00004.json
problem00005
	 language:  en
	 9 candidate authors
	 63 known texts
	 vocabulary size: 12894
	 264 unknown texts
	 24 texts left unattributed
	 answers saved to file answers-problem00005.json
elapsed time: 314.5315752029419


In [222]:
import argparse
import os
import json
import warnings
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from sklearn.preprocessing import LabelEncoder


def eval_measures(gt, pred):
    """Compute macro-averaged F1-scores, macro-averaged precision, 
    macro-averaged recall, and micro-averaged accuracy according the ad hoc
    rules discussed at the top of this file.
    Parameters
    ----------
    gt : dict
        Ground truth, where keys indicate text file names
        (e.g. `unknown00002.txt`), and values represent
        author labels (e.g. `candidate00003`)
    pred : dict
        Predicted attribution, where keys indicate text file names
        (e.g. `unknown00002.txt`), and values represent
        author labels (e.g. `candidate00003`)
    Returns
    -------
    f1 : float
        Macro-averaged F1-score
    precision : float
        Macro-averaged precision
    recall : float
        Macro-averaged recall
    accuracy : float
        Micro-averaged F1-score
    """
    
    actual_authors = list(gt.values())
    
    encoder = LabelEncoder().fit(['<UNK>'] + actual_authors)
    
    text_ids, gold_authors, silver_authors = [], [], []
    for text_id in sorted(gt):
        text_ids.append(text_id)
        gold_authors.append(gt[text_id])
        try:
            silver_authors.append(pred[text_id])
        except KeyError:
            # missing attributions get <UNK>:
            silver_authors.append('<UNK>')

    assert len(text_ids) == len(gold_authors)
    assert len(text_ids) == len(silver_authors)

    # replace non-existent silver authors with '<UNK>':
    silver_authors = [a if a in encoder.classes_ else '<UNK>' 
                      for a in silver_authors]

    gold_author_ints = encoder.transform(gold_authors)
    silver_author_ints = encoder.transform(silver_authors)
    


    # get F1 for individual classes (and suppress warnings):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        labels=list(set(gold_author_ints))
        # Exclude the <UNK> class
       
        for x in labels:
            
            if (x==0):
                labels.remove(x)
        f1 = f1_score(gold_author_ints,
                  silver_author_ints,
                  labels,
                  average='macro')
        precision = precision_score(gold_author_ints,
                  silver_author_ints,
                  labels,
                  average='macro')
        recall = recall_score(gold_author_ints,
                  silver_author_ints,
                  labels,
                  average='macro')
        accuracy = accuracy_score(gold_author_ints,
                  silver_author_ints)

    return f1,precision,recall

def evaluate(ground_truth_file,predictions_file):
    # Calculates evaluation measures for a single attribution problem
    gt = {}
    with open(ground_truth_file, 'r') as f:
        for attrib in json.load(f)['ground_truth']:
            gt[attrib['unknown-text']] = attrib['true-author']

    pred = {}
    with open(predictions_file, 'r') as f:
        for attrib in json.load(f):
            if attrib['unknown-text'] not in pred:
                pred[attrib['unknown-text']] = attrib['predicted-author']
    f1,precision,recall =  eval_measures(gt,pred)
    return round(f1,3), round(precision,3), round(recall,3)

def evaluate_all(path_collection,path_answers,path_out):
    # Calculates evaluation measures for a PAN-18 collection of attribution problems
    infocollection = path_collection+os.sep+'collection-info.json'
    problems = []
    data = []
    with open(infocollection, 'r') as f:
        for attrib in json.load(f):
            problems.append(attrib['problem-name'])
    scores=[];
    for problem in problems[0:5]:
        f1,precision,recall=evaluate(path_collection+os.sep+problem+os.sep+'ground-truth.json',path_answers+os.sep+'answers-'+problem+'.json')
        scores.append(f1)
        data.append({'problem-name': problem, 'macro-f1': round(f1,3), 'macro-precision': round(precision,3), 'macro-recall': round(recall,3)})
        print(str(problem),'Macro-F1:',round(f1,3))
    overall_score=(scores[0]+scores[2]+scores[4])/3
    # Saving data to output files (out.json and evaluation.prototext)
    with open(path_out+os.sep+'out.json', 'w') as f:
        json.dump({'problems': data, 'overall_score': round(overall_score,3)}, f, indent=4, sort_keys=True)
    print('Overall score:', round(overall_score,3))
    prototext='measure {\n key: "mean macro-f1"\n value: "'+str(round(overall_score,3))+'"\n}\n'
    with open(path_out+os.sep+'evaluation.prototext', 'w') as f:
        f.write(prototext)
        
def main():
    i = 'C:\\Users\\90538\\Downloads\\pan19-cross-domain-authorship-attribution-training-dataset-2019-01-23\\pan19-cross-domain-authorship-attribution-training-dataset-2019-01-23'    
    a = "C:\\Users\\90538\\Downloads\\outputs"
    o = "C:\\Users\\90538\\Downloads\\eval"
    evaluate_all(i,a,o)

if __name__ == '__main__':
    main()

problem00001 Macro-F1: 0.718
problem00002 Macro-F1: 0.533
problem00003 Macro-F1: 0.589
problem00004 Macro-F1: 0.34
problem00005 Macro-F1: 0.507
Overall score: 0.605
